# Turismo de Lisboa 

Version 3

Student: Ana Isabel Rocha Moura - 20250168

Date: 26/12/2025 


## Data Preparation 


In [1]:
# Load packages 

import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0 #For consistentncy

import re

# Load datasets
reviews = pd.read_csv("attraction_data_reviews.csv")
metadata = pd.read_csv("attractions_data_metadata.csv")
attractions = pd.read_csv("attractions_list.csv")

### Format Types

In [2]:
#Type of columns in metadata
print(reviews.dtypes)

title                   object
text                    object
rating                   int64
reviewer                object
visitMonth              object
country                 object
visitType               object
contributions          float64
likesNumber              int64
replyText               object
replyDate               object
reviewDate              object
name                    object
tripadvisorId            int64
dateFirstCollection     object
dtype: object


In [3]:
#Type of columns in metadata
print(metadata.dtypes)

tripadvisorId                            int64
name                                    object
location                                object
latitude                               float64
longitude                              float64
dateFirstCollection                     object
ratingGlobal                           float64
reviewCount                            float64
url                                     object
attractionCategory                      object
rankingPositionAttractionCategory      float64
totalRankingUnitsAttractionCategory    float64
dtype: object


In [4]:
#Type of columns in attractions
print(attractions.dtypes)

name              object
url               object
tripadvisor_id     int64
num_reviews        int64
dtype: object


In Reviews dataframe float64 can be replaced by int64 dtype. 
Objects like, title, text, reviwer, country, visit type, replyText, name. 
Substitute visitMonth, likesNumber, replyDate, reviewDate, dateFirstColection to int64 or float64.

### Missing Values In reviews

In [5]:
#Total Number of missing values for every column in reviews
print("\nTotal and Percentage of null values per column")
summary_reviews = pd.DataFrame({
    'Total': reviews.isnull().sum(),
    'Percentage (%)': reviews.isnull().mean()*100
})
print(summary_reviews)


Total and Percentage of null values per column
                     Total  Percentage (%)
title                  463        0.322298
text                     0        0.000000
rating                   0        0.000000
reviewer                11        0.007657
visitMonth           24375       16.967617
country              87226       60.718661
visitType            29803       20.746088
contributions          521        0.362672
likesNumber              0        0.000000
replyText            94321       65.657543
replyDate            94321       65.657543
reviewDate               0        0.000000
name                     0        0.000000
tripadvisorId            0        0.000000
dateFirstCollection      0        0.000000


In [6]:
#Total of values in reviews 
print(reviews.count())

title                  143193
text                   143656
rating                 143656
reviewer               143645
visitMonth             119281
country                 56430
visitType              113853
contributions          143135
likesNumber            143656
replyText               49335
replyDate               49335
reviewDate             143656
name                   143656
tripadvisorId          143656
dateFirstCollection    143656
dtype: int64


#### Constributions

In [7]:
#Fill missing Values in Contributions with 0
reviews['contributions'] = reviews['contributions'].fillna(0).astype('int64')

In [8]:
#View Results
print(reviews['contributions'].head(10))
print(reviews['contributions'].tail(10))

0     1
1    13
2    46
3     2
4     2
5     3
6     1
7     8
8     5
9     2
Name: contributions, dtype: int64
143646     17
143647      1
143648      1
143649      4
143650     44
143651      9
143652      1
143653      7
143654    154
143655      2
Name: contributions, dtype: int64


#### Title

In [9]:
#Fill missing Values in Title with No review title
reviews['title'] = reviews['title'].fillna("no_review_title")

In [10]:
#View Results
reviews['title'].value_counts()

title
Excellent                                                        852
Great experience                                                 807
Highly recommended                                               615
Great tour                                                       586
no_review_title                                                  463
                                                                ... 
EXCEPTIONAL tour with wonderful guide and driver                   1
A great way to start your stay in Lisbon!                          1
Great Insider Knowledge of Lisbon History and Avoiding Stairs      1
Amazing walking tour with Elsa                                     1
Well structured and enjoyable                                      1
Name: count, Length: 102237, dtype: int64

#### Reviewer 

In [11]:
#Fill missing Values in Reviewer with no reviewer
reviews['reviewer'] = reviews['reviewer'].fillna("no_reviewer")

In [12]:
#View Results
reviews['reviewer'].value_counts()

reviewer
Thomas V         208
claudio d        125
Brad              91
YB1972            84
David S           76
                ... 
efa622             1
imalldunn          1
SusanB81876        1
Freedom820051      1
Brucer1011         1
Name: count, Length: 85947, dtype: int64

Check for duplicates in reviewer and text

#### Reply Text 

In [13]:
#Fill missing Values in replyText with no reply text
reviews['replyText'] = reviews['replyText'].fillna("no_reply_text")

In [14]:
#View Results
reviews['replyText'].value_counts()

replyText
no_reply_text                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

#### Reply Date

In [15]:
reviews['replyDate'] = pd.to_datetime(reviews['replyDate'])
print(reviews['replyDate'].min())

2023-01-01 00:00:00


In [16]:
#Fill missing Values in replyDate with 2023-01-01 00:00:00
reviews['replyDate'] = reviews['replyDate'].fillna("2023-01-01")

In [17]:
#View Results
reviews['replyDate'].value_counts()

replyDate
2023-01-01    94322
2024-12-16      500
2023-10-31      456
2025-05-23      402
2025-05-05      283
              ...  
2025-08-22        1
2025-08-23        1
2023-01-22        1
2023-02-04        1
2025-08-15        1
Name: count, Length: 961, dtype: int64

Check volume of replyDate and if it influences rating

#### Visit Month 

In [18]:
#Check for missing values in Review Date
print(reviews['reviewDate'].isna().sum())

0


In [19]:
print(reviews['visitMonth'])
print(reviews['reviewDate'])

0         2023-09
1         2025-04
2         2025-02
3         2024-11
4         2024-12
           ...   
143651    2023-05
143652        NaN
143653        NaN
143654    2025-05
143655    2025-06
Name: visitMonth, Length: 143656, dtype: object
0         2023-09-27
1         2025-04-26
2         2025-02-18
3         2024-12-23
4         2024-12-13
             ...    
143651    2023-05-15
143652    2023-02-13
143653    2023-07-16
143654    2025-05-22
143655    2025-07-01
Name: reviewDate, Length: 143656, dtype: object


In [20]:
#Replace missing data in visitMonth by Review Date 

#reviewDate and visitMonth in datetime format
reviews['reviewDate'] = pd.to_datetime(reviews['reviewDate'])
reviews['visitMonth'] = pd.to_datetime(reviews['visitMonth']) #Here data will have 01 in the end (ex: 2025-02-01)

#Replace string (YYYY-MM) from the reviewDate
replace_values = reviews['reviewDate'].dt.strftime('%Y-%m')

# Fill missing visitMonths with reviewDate
reviews['visitMonth'] = reviews['visitMonth'].fillna(replace_values)


In [21]:
#Check for missing values in visit month
missing_after = reviews['visitMonth'].isnull().sum()
print(f"Missing values in visitMonth: {missing_after}")

Missing values in visitMonth: 0


In [22]:
#View Results
reviews['visitMonth'].value_counts()

visitMonth
2025-05-01    7010
2024-08-01    6737
2024-09-01    6314
2024-05-01    6276
2025-04-01    6107
2024-10-01    5953
2024-07-01    5751
2024-06-01    5494
2023-09-01    5484
2024-04-01    5333
2025-03-01    5006
2025-06-01    4966
2023-08-01    4854
2025-07-01    4838
2023-10-01    4694
2024-03-01    4660
2023-07-01    4595
2024-11-01    4581
2023-05-01    4406
2023-04-01    4317
2023-06-01    3830
2024-12-01    3792
2025-02-01    3631
2023-11-01    3570
2023-03-01    3425
2024-02-01    3325
2025-01-01    3192
2023-12-01    3160
2024-01-01    2672
2023-02-01    2461
2023-01-01    2317
2022-12-01     361
2022-11-01     113
2022-10-01      85
2022-08-01      78
2022-09-01      76
2022-06-01      42
2022-07-01      41
2022-05-01      37
2022-04-01      36
2022-03-01      20
2022-02-01      12
2019-07-01       1
2019-01-01       1
2017-06-01       1
2025-08-01       1
Name: count, dtype: int64

#### Visit Type

In [23]:
#Fill missing Values in Visit Type with 
reviews['visitType'] = reviews['visitType'].fillna("no_information")

In [24]:
#View Results
reviews['visitType'].value_counts()

visitType
Couples           41691
Family            33362
no_information    29803
Friends           26166
Solo              11341
Business           1293
Name: count, dtype: int64

In [25]:
#View Results
reviews['visitType'].head()

0       Solo
1    Friends
2    Couples
3    Couples
4    Couples
Name: visitType, dtype: object

#### Country

In [26]:
#See unique value
unique_countries = reviews['country'].unique()
print(f"Total unique countries: {len(unique_countries)}")

#Top 30 to see variations
print(reviews['country'].value_counts().head(30))

Total unique countries: 12072
country
Lisbon, Portugal           1630
London, UK                 1183
Madrid, Spain               866
New York City, NY           640
Barcelona, Spain            548
Toronto, Canada             534
Rome, Italy                 496
Milan, Italy                424
Chicago, IL                 391
Paris, France               377
Sao Paulo, SP               324
Los Angeles, CA             319
Melbourne, Australia        307
Boston, MA                  301
Dublin, Ireland             297
San Francisco, CA           287
Buenos Aires, Argentina     270
Atlanta, GA                 261
Sydney, Australia           257
Washington DC, DC           252
Oakland, CA                 250
Rio de Janeiro, RJ          236
Valencia, Spain             222
Miami, FL                   220
Hong Kong, China            212
Montreal, Canada            211
Seattle, WA                 194
San Diego, CA               192
Turin, Italy                190
United Kingdom              184
Na

In [27]:
#Convert the entire column to lowercase
reviews['country'] = reviews['country'].str.lower().str.strip()

print(reviews['country'].head(10))

0             NaN
1             NaN
2             NaN
3             NaN
4             NaN
5             NaN
6             NaN
7             NaN
8    glenview, il
9             NaN
Name: country, dtype: object


In [28]:
#Remove acents and other caracteres 

import unicodedata
import re

def remove_accents_and_clean(text):
    if pd.isna(text) or str(text).lower() == 'unknown':
        return 'unknown'
    
    #Normalize unicode characters
    text = unicodedata.normalize('NFD', str(text))
    text = "".join([c for c in text if unicodedata.category(c) != 'Mn'])
    
    #Remove non-alphabetic characters (keep spaces and commas)
    text = re.sub(r'[^a-zA-Z\s,]', '', text)
    
    #Clean double spaces and return small caps
    text = text.lower()
    
    #Clean extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    #If string is empty after cleaning, return unknown
    if not text or text == 'nan' or text == 'world':
        return 'unknown'
        
    return text

#Apply
reviews['country_inter'] = reviews['country'].apply(remove_accents_and_clean)
print(reviews['country_inter'].value_counts().head(10))

country_inter
unknown              87261
lisbon, portugal      1630
london, uk            1183
madrid, spain          866
new york city, ny      640
barcelona, spain       548
toronto, canada        534
rome, italy            496
milan, italy           424
chicago, il            392
Name: count, dtype: int64


In [29]:
#Dictionary for countries
country_standardization_map = {
    #Variations for United States
    'usa': 'united states',
    'states': 'united states',
    'u.s.a.': 'united states',
    'u.s': 'united states',
    'us': 'united states',
    'america': 'united states',
    'united states': 'united states',
    'alabama': 'united states',
    'alaska': 'united states',
    'atlanta': 'united states',
    'arizona': 'united states',
    'arkansas': 'united states',
    'boston': 'united states',
    'california': 'united states',
    'colorado': 'united states',
    'connecticut': 'united states',
    'chicago': 'united states',
    'delaware': 'united states',
    'florida': 'united states',
    'georgia': 'united states', 
    'hawaii': 'United States',
    'idaho': 'united states',
    'illinois': 'united states', 
    'indiana': 'united states',
    'iowa': 'united states',
    'kansas': 'united states',
    'kentucky': 'united states',
    'los angeles': 'united states',
    'louisiana': 'united states',
    'maine': 'united states',
    'maryland': 'united states',
    'massachusetts': 'united states',
    'michigan': 'united states',
    'minnesota': 'united states',
    'mississippi': 'united states',
    'missouri': 'united states',
    'montana': 'united states',
    'nebraska': 'united states',
    'nevada': 'united states',
    'new hampshire': 'united states',
    'new jersey': 'united states',
    'new mexico':'united states',
    'new york': 'united states',
    'north carolina': 'united states',
    'north dakota': 'united states',
    'ohio': 'united states',
    'oklahoma': 'united states',
    'oregon': 'united states',
    'pennsylvania': 'united states',
    'providence': 'united states',
    'raleigh': 'united states',
    'rhode island': 'united states',
    'san francisco': 'united states',
    'seatle': 'united states',
    'seattle': 'united states',
    'south carolina': 'united states',
    'south dakota': 'united states',
    'tennessee': 'united states',
    'texas': 'united states',
    'utah': 'united states',
    'vermont': 'united states',
    'virginia': 'united states',
    'washington': 'united states',
    'west virginia': 'united states',
    'wisconsin': 'united states',
    'wyoming': 'united states',
    'district of columbia': 'united states',
    'al': 'united states',
    'ak': 'united states',
    'az': 'united states',
    'ar': 'united states', 
    'ca': 'united states',
    'co': 'united states',
    'ct': 'united states',
    'de': 'united states', 
    'fl': 'united states',
    'ga': 'united states',
    'hi': 'united states',
    'id': 'united states', 
    'il': 'united states',
    'in': 'united states',
    'ia': 'united states',
    'ks': 'united states', 
    'ky': 'united states',
    'la': 'united states',
    'me': 'united states',
    'md': 'united states', 
    'ma': 'united states',
    'mi': 'united states',
    'mn': 'united states',
    'ms': 'united states', 
    'mo': 'united states',
    'mt': 'united states',
    'ne': 'united states',
    'nv': 'united states', 
    'nh': 'united states',
    'nj': 'united states',
    'nm': 'united states',
    'ny': 'united states', 
    'nyc': 'united states', 
    'nc': 'united states',
    'nd': 'united states',
    'oh': 'united states',
    'ok': 'united states', 
    'or': 'united states',
    'pa': 'united states',
    'ri': 'united states',
    'sc': 'united states', 
    'sd': 'united states',
    'tn': 'united states',
    'tx': 'united states',
    'ut': 'united states', 
    'vt': 'united states',
    'va': 'united states',
    'wa': 'united states',
    'wv': 'united states', 
    'wi': 'united states',
    'wy': 'united states',
    'dc': 'united states',
    'arlington': 'united states',
    'franklin': 'united states',
    'mathews': 'united states',
    'matthews': 'united states',
    'papillion': 'united states',
    'minneapolis': 'united states',
    'cincinnati': 'united states',
    'washington dc': 'united states',
    'st louis': 'united states',
    'orlando': 'united states',
    'carrollton': 'united states',
    'spokane': 'united states',
    'new england': 'united states',
    'dallas': 'united states',
    'houston': 'united states',
    'wayland': 'united states',
    'federal district': 'united states',
    'miami': 'united states',
    'oakland': 'united states',
    'anaheim': 'united states',
    'san diego': 'united states',
    'cleveland': 'united states',
    'baltimore': 'united states',
    'los gatos': 'united states',
    'gettysburg': 'united states',
    'las vegas': 'united states',
    'boise': 'united states',
    'chevy chase': 'united states',
    'sterling heights': 'united states',
    'phoenix': 'united states',
    'kauai': 'united states',
    'charlottesville': 'united states',
    'boca raton': 'united states',
    'savannah': 'united states',
    'south florida': 'united states',
    'theresa sutter': 'united states',
    'key largo': 'united states',
    'new york city': 'united states',
    'tampa': 'united states',
    "new york": "united states",
    "new york city": "united states",
    "nyc": "united states",
    "los angeles": "united states",
    "la": "united states",
    "chicago": "united states", "jlgchicago": "united states", "chicago< illionois": "united states",
    "houston": "united states",
    "phoenix": "united states",
    "philadelphia": "united states", "phila": "united states",
    "philly": "united states",
    "san antonio": "united states",
    "san diego": "united states",
    "dallas": "united states",
    "san jose": "united states",
    "austin": "united states",
    "jacksonville": "united states",
    "san francisco": "united states",
    "sf": "united states",
    "columbus": "united states",
    "fort worth": "united states",
    "indianapolis": "united states",
    "charlotte": "united states",
    "seattle": "united states",
    "denver": "united states",
    "washington": "united states",
    "washington dc": "united states",
    "dc": "united states",
    "boston": "united states",
    "el paso": "united states",
    "nashville": "united states",
    "detroit": "united states",
    "oklahoma city": "united states",
    "portland": "united states",
    "las vegas": "united states",
    "vegas": "united states",
    "memphis": "united states",
    "louisville": "united states",
    "baltimore": "united states",
    "milwaukee": "united states",
    "albuquerque": "united states",
    "tucson": "united states",
    "fresno": "united states",
    "sacramento": "united states",
    "mesa": "united states",
    "atlanta": "united states",
    "kansas city": "united states",
    "colorado springs": "united states",
    "miami": "united states",
    "raleigh": "united states",
    "omaha": "united states",
    "long beach": "united states",
    "virginia beach": "united states",
    "oakland": "united states",
    "minneapolis": "united states",
    "tulsa": "united states",
    "arlington": "united states",
    "new orleans": "united states",
    'asheville': 'united states',
    'hollywood': 'united states', 'n hollywood': 'united states',
    'wisconsin dells': 'united states',
    'west coast': 'united states', 
    'shaker heights': 'united states', 
    'north america': 'united states', 
    'brookfield ct': 'united states', 'brookfield': 'united states',
    'mesquite nv': 'united states', 
    'north america': 'united states', 
    'georgetown': 'united states', 
    'south coast mass': 'united states', 
    'healdsburg': 'united states', 
    'joliet': 'united states', 
    'glendale': 'united states', 
    'fremont mi': 'united states',
    'fremont': 'united states',
    'sfo': 'united states',
    'beachwood': 'united states',
    'lake charles': 'united states',
    'miami beach': 'united states',
    'sf bay area': 'united states',
    'san francisco bay area': 'united states',
    'santa monica': 'united states',
    'berkshire': 'united states',
    'pleasant hill': 'united states', 'pleasant': 'united states',
    'niagara falls': 'united states', 'niagara': 'united states', 'falls': 'united states',
    'brooklyn': 'united states',
    'kalamazoo': 'united states',
    'calabash': 'united states',
    'monroe': 'united states',
    'san ramon': 'united states',
    'alpharetta': 'united states',
    'oskaloosa': 'united states',
    'grand marais': 'united states',


    
    #Variations for United Kingdom
    'uk': 'united kingdom', 
    'u.k.': 'united kingdom', 
    'gb': 'united kingdom',
    'g.b.': 'united kingdom',
    'united kingdom': 'united kingdom',
    'great britain': 'united kingdom',
    'england': 'united kingdom',
    'britain': 'united kingdom',
    'scotland': 'united kingdom',
    'wales': 'united kingdom',
    'northern ireland': 'united kingdom',
    'ireland': 'united kingdom',
    'london': 'united kingdom',
    'manchester': 'united kingdom',
    'birmingham': 'united kingdom',
    'glasgow': 'united kingdom',
    'edinburgh': 'united kingdom',
    'liverpool': 'united kingdom',
    'bristol': 'united kingdom',
    'leeds': 'united kingdom',
    'sheffield': 'united kingdom',
    'cardiff': 'united kingdom',
    'belfast': 'united kingdom',
    'aberdeen': 'united kingdom',
    'newcastle': 'united kingdom',
    'glenrothes': 'united kingdom', 
    'fife': 'united kingdom',
    'yorkshire': 'united kingdom',
    'lancashire': 'united kingdom',
    'devon': 'united kingdom',
    'cornwall': 'united kingdom',
    'hertfordshire': 'united kingdom',
    'kent': 'united kingdom',
    'bellingham': 'united kingdom',
    'guildford': 'united kingdom',
    'southport': 'united kingdom',
    'yaxley': 'united kingdom',
    'solihull': 'united kingdom',
    'st paul': 'united kingdom',
    'bridgend': 'united kingdom',
    'bath': 'united kingdom',
    'birmingham': 'united kingdom',
    'bradford': 'united kingdom',
    'brighton & hove': 'united kingdom',
    'brighton and hove': 'united kingdom',
    'bristol': 'united kingdom',
    'cambridge': 'united kingdom',
    'canterbury': 'united kingdom',
    'chelmsford': 'united kingdom',
    'chester': 'united kingdom',
    'colchester': 'united kingdom',
    'coventry': 'united kingdom',
    'derby': 'united kingdom',
    'doncaster': 'united kingdom',
    'durham': 'united kingdom',
    'ely': 'united kingdom',
    'exeter': 'united kingdom',
    'gloucester': 'united kingdom',
    'hull': 'united kingdom',
    'kingston upon hull': 'united kingdom',
    'leeds': 'united kingdom',
    'leicester': 'united kingdom',
    'liverpool': 'united kingdom',
    'london': 'united kingdom',
    'city of london': 'united kingdom',
    'manchester': 'united kingdom',
    'milton keynes': 'united kingdom',
    'newcastle': 'united kingdom',
    'newcastle upon tyne': 'united kingdom',
    'norwich': 'united kingdom',
    'nottingham': 'united kingdom',
    'oxford': 'united kingdom',
    'peterborough': 'united kingdom',
    'plymouth': 'united kingdom',
    'portsmouth':'united kingdom','portsmouth uk':'united kingdom',
    'preston': 'united kingdom',
    'salford': 'united kingdom',
    'salisbury': 'united kingdom',
    'sheffield': 'united kingdom',
    'southampton': 'united kingdom',
    'southend-on-sea': 'united kingdom',
    'st albans': 'united kingdom',
    'stoke-on-trent': 'united kingdom',
    'sunderland':'united kingdom',
    'truro': 'united kingdom',
    'wakefield': 'united kingdom',
    'wells': 'united kingdom',
    'westminster': 'united kingdom',
    'winchester': 'united kingdom',
    'wolverhampton': 'united kingdom',
    'worcester': 'united kingdom',
    'york': 'united kingdom',
    'surrey': 'united kingdom',
    'hook': 'united kingdom',
    'eastbourne': 'united kingdom',
    'lincoln': 'united kingdom', 
    'richmond': 'united kingdom',
    'norfolk': 'united kingdom',
    'corringham': 'united kingdom',
    'tipton west midlands': 'united kingdom',
    'sevenoaks': 'united kingdom',
    'crowthorne': 'united kingdom',
    'lichfield': 'united kingdom',
    'dungloe': 'united kingdom',
    'essex': 'united kingdom',
    'shropshire': 'united kingdom',
    'dundee': 'united kingdom',
    'somerset': 'united kingdom',


    #Variations for Portugal 
    'portugal continent': 'portugal',
    'pt': 'portugal', 
    'portugal': 'portugal',
    'por': 'portugal', 
    'lisbon': 'portugal',
    'lisboa': 'portugal',
    'porto': 'portugal',
    'faro': 'portugal',
    'coimbra': 'portugal',
    'braga': 'portugal',
    'aveiro': 'portugal',
    'setúbal': 'portugal',
    'guimarães': 'portugal',
    'madeira': 'portugal',
    'azores': 'portugal',
    'açores': 'portugal',
    'algarve': 'portugal',
    'alentejo': 'portugal',
    'douro': 'portugal',
    'minho': 'portugal',
    'leiria': 'portugal',
    'viseu': 'portugal',
    'cascais': 'portugal',
    'almada': 'portugal',
    'cintra': 'portugal',
    'sintra': 'portugal',
    '2745-102': 'portugal',
    'alcochete': 'portugal',
    'carnaxide': 'portugal',
    'oeiras': 'portugal', 'oeiras valley': 'portugal',
    'queijas': 'portugal',
    'estoril': 'portugal',
    'seixal': 'portugal',
    'massamá': 'portugal', 'massama': 'portugal',

    
    
    #Variations for Germany
    'de': 'germany',
    'frg': 'germany',
    'germany': 'germany',
    'deutschland': 'germany',
    'berlin': 'germany',
    'hamburg': 'germany',
    'munich': 'germany',
    'münchen': 'germany',
    'cologne': 'germany',
    'köln': 'germany',
    'frankfurt': 'germany',
    'stuttgart': 'germany',
    'dusseldorf': 'germany',
    'düsseldorf': 'germany',
    'dortmund': 'germany',
    'essen': 'germany',
    'leipzig': 'germany',
    'dresden': 'germany',
    'hanover': 'germany',
    'bavaria': 'germany',
    'bayern': 'germany',
    'baden-württemberg': 'germany',
    'north rhine-westphalia': 'germany',
    'niedersachsen': 'germany',
    'hesse': 'germany',
    'hessen': 'germany',
    'rhineland-palatinate': 'germany',
    'brandenburg': 'germany',
    'erlangen': 'germany',
    'hannover': 'germany',
    'aschaffenburg': 'germany',


    #Variations for Spain
    'es': 'spain',
    'spain': 'spain',
    'espana': 'spain', 
    'españa': 'spain',
    'madrid': 'spain',
    'barcelona': 'spain',
    'valencia': 'spain',
    'seville': 'spain',
    'sevilla': 'spain',
    'zaragoza': 'spain',
    'malaga': 'spain',
    'málaga': 'spain',
    'bilbao': 'spain',
    'alicante': 'spain',
    'cordoba': 'spain',
    'córdoba': 'spain',
    'andalusia': 'spain',
    'catalonia': 'spain',
    'comunidad de madrid': 'spain',
    'basque country': 'spain',
    'galicia': 'spain',
    'valencian community': 'spain',
    'canary islands': 'spain',
    'balearic islands': 'spain',
    'mallorca': 'spain',
    'ibiza': 'spain',
    'tenerife': 'spain',
    'ct': 'spain', 
    'an': 'spain', 
    'md': 'spain',
    'valladolid': 'spain',
    'gijon': 'spain',
    'gijón': 'spain',
    'leon': 'spain',
    'león': 'spain',
    'granada': 'spain',
    'murcia': 'spain',
    'oviedo': 'spain',
    'palma de mallorca': 'spain',
    'andorra': 'spain', 'andorra la vella': 'spain',
    'orotava': 'spain','orotava39': 'spain',
    'vitoria': 'spain',
    'navarra': 'spain',
    'torre del mar': 'spain',
    'palma nova': 'spain',
    'logroño': 'spain',
    'sant pere de ribes': 'spain',
    'telde': 'spain',
    'sant pere de ribes': 'spain',
    'brivio': 'spain',
    'almeria': 'spain',

    'gibraltar': 'gibraltar', 
    
    

    #Variations for France
    'fr': 'france',
    'francia': 'france',
    'france': 'france',
    'fce': 'france',
    'paris': 'france',
    'marseille': 'france',
    'lyon': 'france',
    'toulouse': 'france',
    'nice': 'france',
    'nantes': 'france',
    'strasbourg': 'france',
    'montpellier': 'france',
    'bordeaux': 'france',
    'lille': 'france',
    'rennes': 'france',
    'ile-de-france': 'france',
    'provence': 'france',
    'côte d’azur': 'france','cote dazur': 'france',
    'brittany': 'france',
    'normandy': 'france',
    'aquitaine': 'france',
    'loire valley': 'france',
    'ile de france': 'france',
    'chalon-sur-saone': 'france',
    'la bedoule': 'france',
    'roquefort-la-bedoule': 'france',
    'auxerre': 'france', 
    'le lamentin': 'france',
    'martinique': 'france',
    'st germain de la grange': 'france',
    'saint-germain-de-la-grange': 'france',
    'mayotte': 'france', 'bry sur marne': 'france',
    'istres': 'france',

    #Variations for Brazil 
    'br': 'brazil',
    'brasil': 'brazil',
    'brazil': 'brazil',
    'rio de janeiro': 'brazil',
    'sao paulo': 'brazil',
    'brasília': 'brazil',
    'brasilia': 'brazil',
    'salvador': 'brazil',
    'fortaleza': 'brazil',
    'belo horizonte': 'brazil',
    'recife': 'brazil',
    'porto alegre': 'brazil',
    'curitiba': 'brazil',
    'ba': 'brazil', 'bahia': 'brazil',
    'ce': 'brazil', 'ceará': 'brazil',
    'pe': 'brazil', 'pernambuco': 'brazil',
    'am': 'brazil', 'amazonas': 'brazil',
    'sp': 'brazil', 'são paulo': 'brazil',
    'rj': 'brazil', 'rio de janeiro (state)': 'brazil',
    'mg': 'brazil', 'minas gerais': 'brazil',
    'pr': 'brazil', 'paraná': 'brazil',
    'rs': 'brazil', 'rio grande do sul': 'brazil',
    'sc': 'brazil', 'santa catarina': 'brazil',
    'df': 'brazil',
    'se': 'brazil',
    'rn': 'brazil', 'natal': 'brazil',
    'state of santa catarina': 'brazil','santa catatina': 'brazil',
    'aracaju': 'brazil', 
    'state of rio grande do sul': 'brazil',
    'joao pessoa': 'brazil', 'pb': 'brazil', 
    'go': 'brazil', 'goiana': 'brazil', 
    'state of sao paulo': 'brazil', 'goiana': 'brazil', 
    'teresina': 'brazil', 'pi': 'brazil', 
    'rio claro': 'brazil', 
    'state of rio de janeiro': 'brazil',
    'state of espirito santo': 'brazil',
    "state of acre": "brazil",
    "state of alagoas": "brazil",
    "state of amapa": "brazil",
    "state of amapá": "brazil",
    "state of amazonas": "brazil",
    "state of bahia": "brazil",
    "state of ceara": "brazil",
    "state of ceará": "brazil",
    "state of espírito santo": "brazil",
    "state of goias": "brazil",
    "state of goiás": "brazil",
    "state of maranhao": "brazil",
    "state of maranhão": "brazil",
    "state of mato grosso": "brazil",
    "state of mato grosso do sul": "brazil",
    "state of minas gerais": "brazil",
    "state of para": "brazil",
    "state of pará": "brazil",
    "state of paraiba": "brazil",
    "state of paraíba": "brazil",
    "state of parana": "brazil",
    "state of paraná": "brazil",
    "state of pernambuco": "brazil",
    "state of piaui": "brazil",
    "state of piauí": "brazil",
    "state of rio de janeiro": "brazil",
    "state of rio grande do norte": "brazil",
    "state of rio grande do sul": "brazil",
    "state of rondonia": "brazil",
    "state of rondônia": "brazil",
    "state of roraima": "brazil",
    "state of sao paulo": "brazil",
    "state of são paulo": "brazil",
    "state of sergipe": "brazil",
    "state of tocantins": "brazil",
    "state of distrito federal": "brazil",
    "porto velho": "brazil", "ro": "brazil",
    "erechim": "brazil",


    #Variations for Italy 
    'ita': 'italy',
    'italy': 'italy',
    'italia': 'italy',
    'it': 'italy',
    'rome': 'italy', 
    'roma': 'italy',
    'milan': 'italy',
    'milano': 'italy',
    'naples': 'italy',
    'napoli': 'italy',
    'turin': 'italy',
    'torino': 'italy',
    'florence': 'italy',
    'firenze': 'italy',
    'venice': 'italy',
    'venezia': 'italy',
    'bologna': 'italy',
    'genoa': 'italy',
    'palermo': 'italy',
    'bari': 'italy',
    'lazio': 'italy',
    'lombardy': 'italy',
    'campania': 'italy',
    'tuscany': 'italy',
    'sicily': 'italy',
    'sardinia': 'italy',
    'veneto': 'italy',
    'piedmont': 'italy',
    'emilia-romagna': 'italy',
    'viterbo': 'italy',
    'bergamo': 'italy',
    'padua': 'italy',
    'verona': 'italy',
    'monleale': 'italy',
    'provincia di monza e brianza': 'italy',
    'fidenza': 'italy',
    'savigliano': 'italy',
    'biassono': 'italy',
    'lecce': 'italy',
    'brescia': 'italy',
    'como': 'italy',
    'salsomaggiore terme': 'italy', 'salsomaggiore': 'italy',
    'monza': 'italy', 
    'castrignano del capo': 'italy',
    'savignano sul rubicone': 'italy',
    'favria': 'italy',
    'saccolongo': 'italy',
    'lecco': 'italy',
    'genova': 'italy',
    'girona': 'italy',
    'ferrara': 'italy',
    'parma': 'italy',
    'catania': 'italy',
    'perugia': 'italy',
    'bolzano': 'italy',

    'marin': 'san marino', 'san marino': 'san marino', 'san marine': 'san marino', 
    'city of san marine': 'san marino', 'marino': 'san marino', 
    

    #Variations for Canada
    'ca': 'canada',
    'canada': 'canada',
    'can': 'canada',
    'alberta': 'canada',
    'british columbia': 'canada',
    'manitoba': 'canada',
    'new brunswick': 'canada',
    'newfoundland and labrador': 'canada',
    'nova scotia': 'canada',
    'ontario': 'canada', 'burlington ontario': 'canada',
    'prince edward island': 'canada',
    'quebec': 'canada',
    'saskatchewan': 'canada',
    'northwest territories': 'canada',
    'nunavut': 'canada',
    'yukon': 'canada',
    'toronto': 'canada',
    'montreal': 'canada',
    'vancouver': 'canada',
    'calgary': 'canada',
    'edmonton': 'canada',
    'ottawa': 'canada',
    'quebec city': 'canada',
    'winnipeg': 'canada',
    'halifax': 'canada',
    'victoria': 'canada',
    'mississauga': 'canada',
    'on': 'canada',
    'qc': 'canada',
    'bc': 'canada',
    'ab': 'canada',
    'mb': 'canada',
    'sk': 'canada',
    'ns': 'canada',
    'nb': 'canada',
    'nl': 'canada',
    'pe': 'canada',
    'yt': 'canada',
    'nt': 'canada',
    'nu': 'canada',
    'newfoundland': 'canada',
    'gatineau': 'canada',
    'kelowna': 'canada',
    'etobicoke': 'canada',
    'montréal': 'canada', 'montreal': 'canada',
    'etobicoke': 'canada',
    'north vancouver': 'canada',
    
    #Variations for Latin Americas
    'argentine': 'argentina', 'buenos aires': 'argentina',
    'argentina': 'argentina', 'rosario': 'argentina',
    'neuquen': 'argentina',
    'mexico': 'mexico', 'mexico city': 'mexico',
    'chihuahua': 'mexico','méxico': 'mexico', 'aguascalientes': 'mexico',
    'colombia': 'colombia', 'bogota': 'colombia','bogotá': 'colombia',
    'chile': 'chile', 'santiago': 'chile',
    'uruguai': 'uruguay', 'montevideo': 'uruguay','ciudad de la costa': 'uruguay',
    'paysandu': 'uruguay','montevideo department': 'uruguay', 'uruguay': 'uruguay',
    'perú': 'peru', 'lima': 'peru',
    'peru': 'peru', 
    'ecuador': 'ecuador', 'quito': 'ecuador','equador': 'ecuador',
    'venezuela': 'venezuela', 'caracas': 'venezuela',
    'puerto rico': 'puerto rico', 
    'san juan': 'puerto rico', 
    'costa rica': 'costa rica', 'san jose': 'costa rica', 
    'guatemala city':'guatemala', 'guatemala': 'guatemala',
    'santo domingo':'dominican republic', 'dominican republic': 'dominican republic',
    'dominican': 'dominican republic', 
    'asuncion': 'paraguay', 'paraguay': 'paraguay', 'paraguai':'paraguay',
    'la paz': 'bolivia', 'bolivia': 'bolivia',
    'aguadulce': 'panama','panama': 'panama', 'panamá': 'panama','panama city': 'panama',
    'tegucigalpa': 'honduras', 'honduras': 'honduras', 'san pedro': 'honduras',
    'managua': 'nicaragua', 'nicaragua': 'nicaragua',
    'san salvador': 'el salvador', 'el salvador': 'el salvador',
    'george town': 'grand cayman', 'grand cayman': 'grand cayman',
    'hagatna': 'mariana islands', 'mariana islands': 'mariana islands',
    'cayo coco': 'cuba', 'cuba': 'cuba', 
    'barbados': 'barbados', 'bridgetown': 'barbados', 
    'trinidad and tobago': 'trinidad and tobago',  'trinidad': 'trinidad and tobago', 
    'tobago': 'trinidad and tobago', 'port of spain': 'trinidad and tobago',
    'willemstad': 'curacao', 'curaçao': 'curacao',
    'reunion island': 'reunion island', 'saint-gilles-les-bains': 'reunion island',
    'st. thomas': 'caribe', 'caribe': 'caribe', 'st. kitts': 'caribe', 'basseterre,': 'caribe', 
    'caribbean': 'caribe',
    'aruba': 'aruba', 
    'providenciales': 'providenciales', 
    'bermuda': 'bermuda', 
    


    #Variations for Austalia
    'austrália': 'australia', 
    'australia': 'australia', 
    'sydney': 'australia', 
    'melbourne': 'australia', 
    'brisbane': 'australia',
    'perth': 'australia',
    'canberra': 'australia',
    'albert park': 'australia',
    'wahroonga': 'australia',
    'albuquerque': 'australia',
    'adelaide': 'australia',
    'sydney': 'australia',
    'brisbane': 'australia',
    'queensland': 'australia',
    'darwin': 'australia',
    'hobart': 'australia',
    'camberra': 'australia',
    'flinders': 'australia',


    #Variations for New Zealand
    'wellington': 'new zealand',
    'new zealand': 'new zealand', 
    'zealand': 'new zealand', 
    'auckland': 'new zealand', 
    'auckland central': 'new zealand', 
    'christchurch': 'new zealand', 
    'dunedin': 'new zealand',


    #Variations for Middle East
    'emirates': 'united arab emirates', 
    'united arab emirates': 'united arab emirates', 
    'abu dhabi': 'united arab emirates',
    'dubai': 'united arab emirates', 
    'uae': 'united arab emirates',
    'israel': 'israel', 'tel aviv': 'israel',
    'jerusalem': 'israel',
    'kuwait':'kuwait', 'kuwait city':'kuwait', 
    'doha': 'qatar', 'qatar': 'qatar', 
    'beirut': 'lebanon', 'lebanon': 'lebanon',
    'riyadh': 'saudi arabia', 'saudi arabia': 'saudi arabia',
    'bahrain': 'bahrain', 'manama': 'bahrain',
    'oman': 'oman', 'sohar': 'oman',
    'kazakhstan':'kazakhstan', 'antananarivo':'kazakhstan', 'almaty':'kazakhstan', 
    'azerbaijan':'azerbaijan', 'baku':'azerbaijan', 

    #Variations for Asia
    'indi': 'india', 
    'india': 'india', 'mumbai': 'india',
    'delhi': 'india', 'pune': 'india',
    'bengaluru': 'india', 
    'nepal': 'nepal', 'gorkha': 'nepal', 'kathmandu': 'nepal', 
    'myanmar': 'myanmar', 'yangon': 'myanmar', 'rangoon': 'myanmar', 
    'japan': 'japan', 'tokyo': 'japan',
    'osaka': 'japan',
    'south korea': 'south korea', 'seoul': 'south korea',
    'kuala lumpur': 'malaysia', 'malaysia': 'malaysia', 
    'bangkok': 'thailand', 'thailand': 'thailand',
    'chine': 'china', 'hong kong': 'hong kong', 
    'beijing': 'china', 'shanghai': 'china',
    'macau': 'china', 'china': 'china',
    'singapore': 'singapore',
    'manila':'philippines', 'philippines': 'philippines',
    'punaauia': 'french polynesia', 'french polynesia': 'french polynesia',
    'jakarta': 'indonesia', 'indonesia': 'indonesia',
    'hanoi':'vietnam', 'vietnam': 'vietnam', 'ho chi minh city': 'vietnam', 
    'taipei': 'taiwan', 'taiwan': 'taiwan', 
    'cambodia': 'cambodia', 'siem reap': 'cambodia', 'phnom penh': 'cambodia',
    'laos': 'laos', 'luang prabang': 'laos', 
    'sri lanka': 'sri lanka', 'colombo': 'sri lanka',
    'new caledonia': 'new caledonia', 


    #Africa
    'south africa': 'south africa', 'cape town': 'south africa','johannesburg': 'south africa',
    'pretoria': 'south africa',
    'mauritius': 'mauritius', 
    'morocco': 'morocco', 'casablanca': 'morocco', 'marrakech': 'morocco',
    'angola': 'angola', 'luanda': 'angola',
    'cape verde': 'cape verde', 'praia': 'cape verde',
    'cairo': 'egypt', 'egypt': 'egypt', 
    'harare': 'zimbabwe', 'zimbabwe':'zimbabwe', 'zimbabué': 'zimbabwe',
    'maputo': 'mozambique', 'benguela': 'mozambique', 'mozambique': 'mozambique', 
    'moçambique': 'mozambique', 'nampula': 'mozambique',
    'douala': 'cameroon', 'cameroon': 'cameroon', 'camaroes': 'cameroon',
    'lagos':'nigeria', 'nigeria': 'nigeria', 
    'egipt': 'egipt', 'alexandria': 'egipt',
    'karen': 'kenya', 'kenya': 'kenya', 'kenia': 'kenya', 'quenia': 'kenya',
    'reunion island': 'reunion island', 'saint-pierre': 'reunion island', 
    'ghana': 'ghana', 'madagascar': 'madagascar', 'antananarivo': 'madagascar',
    'tanzania': 'tanzania', 'dar es salaam region': 'tanzania', 
    'equatorial guinea': 'equatorial guinea', 'guinea': 'equatorial guinea', 'malabo': 'equatorial guinea', 
    'gambia': 'gambia', 'gâmbia': 'gambia', 'kololi': 'gambia', 



    #Europe (South)
    'istanbul': 'turkey', 'türkiye': 'turkey',
    'ancara': 'turkey', 'ankara': 'turkey',
    'izmir': 'turkey',
    'greece': 'greece', 'athens': 'greece',
    'cyprus': 'cyprus', 'nicosia': 'cyprus', 
    'malta':'malta', 'malta island':'malta', 'island of malta':'malta', 
    'monaco':'monaco', 'monaco city': 'monaco', 
    'bosnia and herzegovina': 'bosnia', 'herzegovina': 'bosnia', 'sarajevo': 'bosnia', 

    # Nordic Countries
    'sweden': 'sweden', 'stockholm': 'sweden', 'gothenburg': 'sweden',
    'denmark': 'denmark', 'copenhagen': 'denmark', 'aarhus': 'denmark',
    'norway': 'norway', 'oslo': 'norway', 'bergen': 'norway',
    'finland': 'finland', 'helsinki': 'finland', 
    'the netherlands': 'netherlands', 'haia': 'netherlands',
    'amsterdam': 'netherlands', 'roterdam': 'netherlands',
    'poland': 'poland', 
    'reykjavik': 'iceland', 'iceland': 'iceland',
    'russia': 'russia',
    
    # Central and  Eastern Europe
    'berna': 'switzerland', 'geneva': 'switzerland', 'genebra': 'switzerland', 
    'zurich': 'switzerland', 'zürich': 'switzerland','lausanne': 'switzerland',
    'berne': 'switzerland', 'lucerna': 'switzerland','lugano': 'switzerland',
    'switzerland': 'switzerland', 'schweiz': 'switzerland', 
    'ginevra': 'switzerland',
    'luxembourg': 'luxembourg', 'luxembourg city': 'luxembourg', 'lux': 'luxembourg',
    'triesen':'liechtenstein', 'liechtenstein': 'liechtenstein', 
    'austria': 'austria', 'vienna': 'austria', 'wien': 'austria',
    'sofia':'bulgaria', 'bulgaria': 'bulgaria',
    'poland': 'poland', 'warsaw': 'poland', 'warszawa': 'poland', 'krakow': 'poland',
    'hungary': 'hungary', 'budapest': 'hungary','budapest': 'hungary', 
    'czech republic': 'czech republic', 'prague': 'czech republic', 'praha': 'czech republic',
    'havirov': 'czech republic',
    'belgium': 'belgium', 'brussels': 'belgium', 'bruxelles': 'belgium', 'antwerp': 'belgium',
    'belgique': 'belgium', 'antuerpia': 'belgium', 'liege': 'belgium', 'brussels': 'belgium', 'city of brussels': 'belgium',
    'ireland': 'ireland', 'dublin': 'ireland',
    'bucharest': 'romania', 'romania': 'romania', 'romenia': 'romania', 
    'servia': 'serbia', 'serbia': 'serbia', 'belgrade': 'serbia',
    'ukraine': 'ukraine', 'kyiv': 'ukraine', 'odessa': 'ukraine', 'одесса': 'ukraine', 
    'sofia': 'bulgaria', 'bulgaria': 'bulgaria',
    'estonia':'estonia', 'tallinn': 'estonia',
    'croatia': 'croatia', 'split,': 'croatia', 
    'lithuania': 'lithuania', 'vilnius': 'lithuania',
    'latvia': 'latvia', 'riga': 'latvia', 
    'moldova': 'moldova', 
    'ljubljana': 'slovenia', 'slovenia': 'slovenia', 
    'slovakia':'slovakia', 'slovakia':'slovakia', 
    'yerevan': 'armenia', 'bratislava': 'armenia',
    'belarus': 'belarus', 'mins': 'belarus', 
    'republic of north macedonia': 'north macedonia', 'north macedonia': 'north macedonia',
    'albania': 'albania', 'montgomery': 'albania', 
    
    #Other missed
    'world': 'unknown',
    'europe': 'europe', 'europa': 'europe', 'european union': 'europe', 
    'antarctica': 'antarctica', 'antartida': 'antarctica',


    
}

In [30]:
#Implemente dictionary Changes

def standardize_country_refined(location):
    loc_str = str(location).strip().lower()
    if loc_str in ['unknown', 'nan', 'none', '']:
        return 'unknown'
    
    #Clean string and split
    parts = [p.strip() for p in loc_str.split(',')]
    
    #Search from the last part to the first
    for part in reversed(parts):
        if part in country_standardization_map:
            return country_standardization_map[part]
            
    #If no matched
    return 'not specified'

#Apply to reviews dataset
reviews['country_final'] = reviews['country_inter'].apply(standardize_country_refined)

#Check
print("Standard Country:")
print(reviews['country_final'].value_counts().head(20))

Standard Country:
country_final
unknown           87261
united states     13134
united kingdom     6424
spain              6411
canada             5141
italy              4846
portugal           3472
france             2727
brazil             1776
australia          1391
germany            1260
not specified       896
netherlands         817
switzerland         680
ireland             649
argentina           603
belgium             455
mexico              389
colombia            282
greece              256
Name: count, dtype: int64


In [31]:
# Create 'clean_name' version to compare
reviews['country_clean'] = reviews['country_final'].str.lower().str.replace('the ', '', regex=False).str.strip()

#Check for duplicates on the cleaned version
fuzzy_duplicates = reviews['country_clean'].duplicated().sum()
print(f"Fuzzy name duplicates: {fuzzy_duplicates}")

Fuzzy name duplicates: 143527


In [32]:
# See what is hiding in the 'Not Specified' group
not_specified_examples = reviews[reviews['country_final'] == 'not specified']['country'].value_counts()
print("Not Specified:")
print(not_specified_examples.head(10))
print(not_specified_examples.count())

Not Specified:
country
türkiye                 5
st. thomas              4
fvg                     3
logroño                 3
bodrum city, türkiye    3
w-wa                    3
traveling               3
adana, türkiye          3
chicago< illionois      3
euskalherria            2
Name: count, dtype: int64
799


In [33]:
#Compare 'country' with'country_final'
loss_check = reviews[
    (reviews['country'].notna()) & 
    (reviews['country_final'] == 'unknown')
][['country', 'country_inter']].head(20)

print("Values that became 'Unknown':")
print(loss_check)

Values that became 'Unknown':
        country country_inter
1183      world       unknown
11221  2745-102       unknown
17528     world       unknown
20098     world       unknown
23361     world       unknown
27479     world       unknown
34380     world       unknown
35598         -       unknown
35792     world       unknown
39601     world       unknown
40336     world       unknown
40965     world       unknown
47244     world       unknown
50197     world       unknown
56822     world       unknown
59034     world       unknown
62630     world       unknown
69517     world       unknown
71428     world       unknown
72214     world       unknown


Country Standarization complete. Use country_final

In [34]:
#See real distribution
real_countries = reviews[reviews['country_final'] != 'unknown']

print(real_countries['country_final'].value_counts().head(10))

print(f"\nTotal tourists identified: {len(real_countries)}")

country_final
united states     13134
united kingdom     6424
spain              6411
canada             5141
italy              4846
portugal           3472
france             2727
brazil             1776
australia          1391
germany            1260
Name: count, dtype: int64

Total tourists identified: 56395


In [35]:
reviews['country_final'].count()

np.int64(143656)

#### Final Check for missing Values in Reviews

In [36]:
#Number of missing values for every column
print(reviews.isnull().sum())


title                      0
text                       0
rating                     0
reviewer                   0
visitMonth                 0
country                87226
visitType                  0
contributions              0
likesNumber                0
replyText                  0
replyDate                  0
reviewDate                 0
name                       0
tripadvisorId              0
dateFirstCollection        0
country_inter              0
country_final              0
country_clean              0
dtype: int64


### Missing Values in Metadata

In [37]:
#Total Number of missing values for every column in reviews
print("\nTotal and Percentage of null values per column")
summary_metadata = pd.DataFrame({
    'Total': metadata.isnull().sum(),
    'Percentage (%)': metadata.isnull().mean()*100
})
print(summary_metadata)


Total and Percentage of null values per column
                                     Total  Percentage (%)
tripadvisorId                            0        0.000000
name                                     0        0.000000
location                              1243       68.598234
latitude                              1243       68.598234
longitude                             1243       68.598234
dateFirstCollection                      0        0.000000
ratingGlobal                             1        0.055188
reviewCount                              1        0.055188
url                                      0        0.000000
attractionCategory                       1        0.055188
rankingPositionAttractionCategory        1        0.055188
totalRankingUnitsAttractionCategory      1        0.055188


In [38]:
#Total of values in metadata
print(metadata.count())

tripadvisorId                          1812
name                                   1812
location                                569
latitude                                569
longitude                               569
dateFirstCollection                    1812
ratingGlobal                           1811
reviewCount                            1811
url                                    1812
attractionCategory                     1811
rankingPositionAttractionCategory      1811
totalRankingUnitsAttractionCategory    1811
dtype: int64


#### Location

In [39]:
print(metadata['location'].isnull().sum())
print(metadata['location'].value_counts())

1243
location
Avenida Brasilia, Lisboa 1400-038, Portugal                        4
Largo Trindade Coelho, Lisboa 1200-470, Portugal                   3
Praça do Príncipe Real, Lisboa 1250-184, Portugal                  2
Doca do Bom Sucesso Avenida Brasília, Lisboa 1400-038, Portugal    2
Praça das Novas Nações 44 2 Andar, Lisboa 1170-277, Portugal       2
                                                                  ..
Rua do Norte 91 Bairro Alto, Lisboa 1200-284, Portugal             1
Rua dos Cegos 20, Lisboa 1100-137, Portugal                        1
Rua. Cais de Santarem 23, Lisboa 1100-603, Portugal                1
Rua de Santa Cruz do Castelo, Lisboa 1100-129, Portugal            1
Rua Domingos Sequeira 64b, Lisboa 1350-122, Portugal               1
Name: count, Length: 558, dtype: int64


In [40]:
print(metadata['location'].describe())


count                                             569
unique                                            558
top       Avenida Brasilia, Lisboa 1400-038, Portugal
freq                                                4
Name: location, dtype: object


##### Extract info from the URL

In [41]:
#G code stings from URLs 
metadata['geo_id'] = metadata['url'].str.extract(r'g(\d+)')

#Check unique Geo-IDs in attractions(1812)
print(f"Unique Geo-IDs found: {metadata['geo_id'].nunique()}")
print(metadata['geo_id'].value_counts().head(10))


Unique Geo-IDs found: 1
geo_id
189158    1812
Name: count, dtype: int64


In [42]:
#Extract neighborhoods from valid adresses 

#Valid rows in location 
valid_addresses = metadata[metadata['location'].notnull()]['location']

#Count the most common words in addresses to identify neighborhood names
from collections import Counter
import re

words = " ".join(valid_addresses).lower()
#Remove 'lisboa', 'portugal', and numbers
words = re.sub(r'lisboa|portugal|\d+', '', words)
common_words = Counter(words.split()).most_common(20)

print("Most common words in valid addresses (potential neighborhoods):")
for word, count in common_words:
    if len(word) > 3: # Ignore short words (ex:'rua', 'dos')
        print(f"{word}: {count}")

Most common words in valid addresses (potential neighborhoods):
largo: 73
avenida: 68
praça: 34
bairro: 26
alto,: 24
santo: 18


In [43]:
#Target Neighborhoods based on word list and Lisbon geography
target_neighborhoods = [
    'Alfama', 'Bairro Alto', 'Belem', 'Baixa', 'Chiado', 
    'Principe Real', 'Avenida da Liberdade', 'Parque das Nacoes', 
    'Graca', 'Castelo', 'Alcantara'
]

#Function to look for these names in the Attraction Name or existing Location string
def rescue_neighborhood(row):
    # Check the Name and the Location (filling NaNs with empty string)
    search_text = f"{str(row['name'])} {str(row['location'])}".lower()
    
    for area in target_neighborhoods:
        if area.lower() in search_text:
            return area
    return np.nan 

#Apply rescue
metadata['neighborhood_extracted'] = metadata.apply(rescue_neighborhood, axis=1)

#Check Results 
rescued_count = metadata['neighborhood_extracted'].notnull().sum()
print(f"Neighborhoods identified or rescued: {rescued_count} out of 1812")
print("\nTop Rescued Neighborhoods:")
print(metadata['neighborhood_extracted'].value_counts())

Neighborhoods identified or rescued: 109 out of 1812

Top Rescued Neighborhoods:
neighborhood_extracted
Bairro Alto             27
Chiado                  17
Belem                   15
Alfama                  11
Baixa                    9
Parque das Nacoes        8
Castelo                  7
Avenida da Liberdade     5
Graca                    4
Principe Real            3
Alcantara                3
Name: count, dtype: int64


In [44]:
#Deep Search for missing null values

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

#Initialize the locator 
geolocator = Nominatim(user_agent="lisbon_research_project")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

#Select rows that are still NaN for neighborhood
missing_mask = metadata['neighborhood_extracted'].isna()
to_rescue = metadata[missing_mask].head(50) 

print(f"Attempt rescue for {len(to_rescue)} rows...")

def deep_search(row):
    try:
        # Create a search query: "Attraction Name, Lisbon, Portugal"
        query = f"{row['name']}, Lisbon, Portugal"
        location = geolocator.geocode(query, addressdetails=True, timeout=10)
        
        if location and 'address' in location.raw:
            address = location.raw['address']
            # Look for typical neighborhood fields in the response
            return address.get('suburb') or address.get('neighbourhood') or address.get('city_district')
    except:
        return None
    return None

#Search
metadata.loc[to_rescue.index, 'neighborhood_extracted'] = to_rescue.apply(deep_search, axis=1)

#Check results
new_rescues = metadata['neighborhood_extracted'].notnull().sum()
print(f"Total rescued so far: {new_rescues}")

Attempt rescue for 50 rows...
Total rescued so far: 126


In [45]:
#Bulk processing
#Identify remaining rows as NaN
remaining_missing = metadata[metadata['neighborhood_extracted'].isna()]

print(f"Starting bulk rescue for {len(remaining_missing)} rows.")

#Process info
metadata.loc[remaining_missing.index, 'neighborhood_extracted'] = remaining_missing.apply(deep_search, axis=1)

#Final Check
final_count = metadata['neighborhood_extracted'].notnull().sum()
print(f"Total rescued: {final_count} out of 1812")

#View results
print("\nNew Geographic Distribution:")
print(metadata['neighborhood_extracted'].value_counts().head(15))


Starting bulk rescue for 1686 rows.
Total rescued: 491 out of 1812

New Geographic Distribution:
neighborhood_extracted
Santa Maria Maior       102
Bairro Alto              43
Santa Catarina           41
Santo António            26
Prazeres                 24
Chiado                   17
Alfama                   16
Campo Pequeno            16
Parque das Nações        16
Belem                    15
Belém                    14
Graça                    13
Santa Isabel             11
Baixa                    11
São Francisco Xavier      9
Name: count, dtype: int64


In [46]:
#Clean location information
metropolitan_map = {
    # City of Lisbon
    '1000': 'Lisbon (Avenidas Novas)',
    '1050': 'Lisbon (Saldanha/Campolide)',
    '1070': 'Lisbon (Campolide/Amoreiras)',
    '1100': 'Lisbon (Alfama/Baixa/Castelo)',
    '1150': 'Lisbon (Arroios/Pena)',
    '1170': 'Lisbon (Graca/Anjos)',
    '1200': 'Lisbon (Bairro Alto/Chiado)',
    '1250': 'Lisbon (Principe Real/Rato)',
    '1300': 'Lisbon (Alcantara/Ajuda)',
    '1350': 'Lisbon (Estrela/Campo de Ourique)',
    '1400': 'Lisbon (Belem/Restelo)',
    '1500': 'Lisbon (Benfica)',
    '1600': 'Lisbon (Alvalade/Carnide)',
    '1700': 'Lisbon (Alvalade/Aeroporto)',
    '1750': 'Lisbon (Lumiar/Ameixoeira)',
    '1800': 'Lisbon (Olivais)',
    '1900': 'Lisbon (Chelas/Marvila)',
    '1990': 'Lisbon (Parque das Nacoes)',

    #Nearby Metropolitan Area 
    '2500': 'Caldas da Rainha', 
    '2600': 'Vila Franca de Xira',
    '2670': 'Loures',
    '2675': 'Odivelas',
    '2685': 'Sacavem',
    '2700': 'Amadora',
    '2710': 'Sintra (Historical Center)',
    '2725': 'Mem Martins / Algueirao',
    '2735': 'Agualva-Cacem',
    '2745': 'Queluz',
    '2750': 'Cascais (Center)',
    '2765': 'Estoril',
    '2770': 'Paco de Arcos',
    '2775': 'Parede',
    '2780': 'Oeiras',
    '2790': 'Carnaxide',
    '2795': 'Linda-a-Velha',
    '2800': 'Almada (Center)',
    '2810': 'Laranjeiro / Feijo',
    '2820': 'Charneca da Caparica',
    '2825': 'Costa da Caparica',
    '2830': 'Barreiro',
    '2840': 'Seixal',
    '2855': 'Corroios',
    '2890': 'Alcochete',
    '2900': 'Setubal',
    '2950': 'Palmela'
}

In [47]:
#Extract Postal Code from the original location column
metadata['pc4_original'] = metadata['location'].str.extract(r'(\d{4})')

#If avaliable, extract postal code from the Geopy results
metadata['pc4_rescued'] = metadata['neighborhood_extracted'].str.extract(r'(\d{4})')

#Combine pc4 original with pc4_combined
metadata['pc4_combined'] = metadata['pc4_original'].fillna(metadata['pc4_rescued'])

#Apply Metropolitan Map
metadata['zone_final'] = metadata['pc4_combined'].map(metropolitan_map).fillna('Location NOT Avaliable')

#Check result
print("Final Geographic Distribution:")
print(metadata['zone_final'].value_counts())

Final Geographic Distribution:
zone_final
Location NOT Avaliable               1285
Lisbon (Bairro Alto/Chiado)           154
Lisbon (Alfama/Baixa/Castelo)         113
Lisbon (Principe Real/Rato)            46
Lisbon (Alcantara/Ajuda)               28
Lisbon (Arroios/Pena)                  26
Lisbon (Saldanha/Campolide)            25
Lisbon (Belem/Restelo)                 21
Lisbon (Alvalade/Aeroporto)            20
Lisbon (Graca/Anjos)                   14
Lisbon (Benfica)                       14
Lisbon (Parque das Nacoes)             12
Lisbon (Avenidas Novas)                11
Lisbon (Campolide/Amoreiras)            9
Lisbon (Estrela/Campo de Ourique)       8
Lisbon (Alvalade/Carnide)               8
Lisbon (Chelas/Marvila)                 6
Lisbon (Lumiar/Ameixoeira)              4
Corroios                                2
Lisbon (Olivais)                        2
Estoril                                 1
Sintra (Historical Center)              1
Sacavem                           

In [48]:
#Mix Geopy names and with Metropolitan dict
name_to_zone_bridge = {
    'Santa Maria Maior': 'Lisbon (Alfama/Baixa/Castelo)',
    'Misericórdia': 'Lisbon (Bairro Alto/Chiado)',
    'Santo António': 'Lisbon (Avenida da Liberdade)',
    'Arroios': 'Lisbon (Arroios/Pena)',
    'Estrela': 'Lisbon (Estrela/Campo de Ourique)',
    'Belém': 'Lisbon (Belem/Restelo)',
    'Belem': 'Lisbon (Belem/Restelo)',
    'Alcântara': 'Lisbon (Alcantara/Ajuda)',
    'Parque das Nações': 'Lisbon (Parque das Nacoes)',
    'Avenidas Novas': 'Lisbon (Avenidas Novas)',
    'Sintra': 'Sintra (Historical Center)',
    'Cascais': 'Cascais (Center)',
    'Almada': 'Almada (Center)'
}

#Update the zone_final column
def final_geographic_fix(row):
    if row['zone_final'] != 'Location NOT Avaliable':
        return row['zone_final']
    
    #look at the Geopy name (neighborhood_extracted)
    geo_name = str(row['neighborhood_extracted'])
    for key, zone in name_to_zone_bridge.items():
        if key.lower() in geo_name.lower():
            return zone
            
    return 'Location NOT Avaliable'

#Fix
metadata['zone_final'] = metadata.apply(final_geographic_fix, axis=1)

#Count
print("Final Geographic Distribution:")
print(metadata['zone_final'].value_counts())

Final Geographic Distribution:
zone_final
Location NOT Avaliable               1205
Lisbon (Alfama/Baixa/Castelo)         156
Lisbon (Bairro Alto/Chiado)           154
Lisbon (Principe Real/Rato)            46
Lisbon (Belem/Restelo)                 37
Lisbon (Alcantara/Ajuda)               28
Lisbon (Arroios/Pena)                  26
Lisbon (Saldanha/Campolide)            25
Lisbon (Parque das Nacoes)             23
Lisbon (Alvalade/Aeroporto)            20
Lisbon (Benfica)                       14
Lisbon (Graca/Anjos)                   14
Lisbon (Avenidas Novas)                11
Lisbon (Campolide/Amoreiras)            9
Lisbon (Estrela/Campo de Ourique)       8
Lisbon (Alvalade/Carnide)               8
Lisbon (Avenida da Liberdade)           7
Lisbon (Chelas/Marvila)                 6
Lisbon (Lumiar/Ameixoeira)              4
Sintra (Historical Center)              4
Corroios                                2
Lisbon (Olivais)                        2
Estoril                           

In [49]:
#Fix code

def final_dictionary_cleanup(row):
    # Get available text for attraction
    name = str(row['name']).lower()
    location = str(row.get('location', '')).lower()
    current_zone = str(row.get('zone_final', '')).lower()
    
    #Combine everything into one search string
    text = f"{name} {location} {current_zone}"

    #Lisbon Greater Area
    if any(w in text for w in ['sintra', 'pena', 'regaleira', 'mouros', 'monserrate', 'queluz', 'cabo da roca']): return 'Sintra'
    if any(w in text for w in ['cascais', 'estoril', 'boca do inferno', 'guia', 'tamariz']): return 'Cascais/Estoril'
    if any(w in text for w in ['almada', 'cristo rei', 'caparica']): return 'Almada'

    #Lisbon City
    if any(w in text for w in ['alfama', 'castelo', 'sé', 'fama']): return 'Lisbon (Alfama/Castelo)'
    if any(w in text for w in ['baixa', 'rossio', 'comercio', 'augusta', 'restauradores', 'municipio', 'figueira', 'município', 'santa justa']): return 'Lisbon (Baixa)'
    if any(w in text for w in ['chiado', 'carmo', 'garrett', 'camões', 'camoes', 'bertrand']): return 'Lisbon (Chiado)'
    if any(w in text for w in ['bairro alto', 'misericordia', 'santa catarina', 'bica', 'funicular', 'gloria', 'glória', 'são pedro de alcântara']): return 'Lisbon (Bairro Alto)'
    if any(w in text for w in ['graca', 'graça', 'são vicente', 'panteão', 'panteao', 'sapadores']): return 'Lisbon (Graca/ Sao Vicente)'
    if any(w in text for w in ['belem', 'belém', 'jeronimos', 'torre', 'discoveries']): return 'Lisbon (Belem)'
    if any(w in text for w in ['liberdade', 'tivoli', 'marquês', 'pombal']): return 'Lisbon (Avenida da Liberdade)'
    if any(w in text for w in ['principe real', 'príncipe']): return 'Lisbon (Principe Real)'
    if any(w in text for w in ['alcantara', 'lx factory', 'docas']): return 'Lisbon (Alcantara)'
    if any(w in text for w in ['cais do sodre', 'ribeira', 'pink street', 'mercado da ribeira', 'timeout', 'time out', 'corpo santo']): return 'Lisbon (Cais do Sodre)'
    if any(w in text for w in ['nacoes', 'nações', 'expo', 'oceanario']): return 'Lisbon (Parque das Nacoes)'
    if any(w in text for w in ['avenidas novas', 'saldanha', 'gulbenkian', 'campo pequeno']): return 'Lisbon (Avenidas Novas)'
    if any(w in text for w in ['estrela', 'lapa', 'basilica', 'campo de ourique']): return 'Lisbon (Estrela/Campo de Ourique)'
    if any(w in text for w in ['arroios', 'anjos', 'intendente', 'penha']): return 'Lisbon (Arroios)'

    # No Information provided, only experience
    # If it's a tour and we haven't found a zone, it's almost certainly the Historic Center
    if any(w in text for w in ['tour', 'walking', 'fado', 'experience', 'day trip', 'private trip']):
        return 'Lisbon (Experiences)'

    #Water activities 
    if any(w in text for w in ['river', 'boat', 'tagus', 'tejo', 'sailing', 'cruise']):
        return 'Lisbon (River Experiences)'

    return 'Location NOT Provided'

#Apply logic
metadata['neighborhood_final'] = metadata.apply(final_dictionary_cleanup, axis=1)

#See Results
print("Neigborhood Distribution")
print(metadata['neighborhood_final'].value_counts())

Neigborhood Distribution
neighborhood_final
Location NOT Provided                719
Lisbon (Experiences)                 428
Lisbon (Alfama/Castelo)              162
Lisbon (Chiado)                      156
Lisbon (River Experiences)            60
Lisbon (Belem)                        45
Lisbon (Avenidas Novas)               39
Lisbon (Principe Real)                34
Sintra                                33
Lisbon (Alcantara)                    25
Lisbon (Parque das Nacoes)            21
Lisbon (Graca/ Sao Vicente)           19
Lisbon (Avenida da Liberdade)         17
Lisbon (Baixa)                        15
Lisbon (Estrela/Campo de Ourique)     14
Cascais/Estoril                        9
Lisbon (Bairro Alto)                   7
Almada                                 3
Lisbon (Arroios)                       3
Lisbon (Cais do Sodre)                 3
Name: count, dtype: int64


For final export use the column neighborhood_final for location

#### Rating Global

In [50]:
print(metadata['ratingGlobal'].min())
print(metadata['ratingGlobal'].isnull().sum())

1.0
1


In [51]:
#Fill missing Values in Rating Global with 1
metadata['ratingGlobal'] = metadata['ratingGlobal'].fillna(1).astype('int64')

In [52]:
#View Results
print(metadata['ratingGlobal'].isnull().sum())
print(metadata['ratingGlobal'].value_counts())

0
ratingGlobal
5    824
4    725
3    183
2     42
1     38
Name: count, dtype: int64


#### Review Count


In [53]:
print(metadata['reviewCount'].min())
print(metadata['reviewCount'].isnull().sum())

1.0
1


In [54]:
# Filling review Count with 1
metadata['reviewCount'] = metadata['reviewCount'].fillna(1).astype('int64')


In [55]:
#View Results
print(metadata['reviewCount'].isnull().sum())
print(metadata['reviewCount'].value_counts())

0
reviewCount
1       206
2       122
3        86
5        61
4        58
       ... 
5343      1
1481      1
2439      1
1632      1
695       1
Name: count, Length: 451, dtype: int64


#### Ranking Position Attraction Category

In [56]:
metadata['rankingPositionAttractionCategory'].min()

1.0

In [57]:
# Filling missing ranking Position Attraction Category with 1
metadata['rankingPositionAttractionCategory'] = metadata['rankingPositionAttractionCategory'].fillna(1).astype('int64')

In [58]:
#View Results
metadata['rankingPositionAttractionCategory'].value_counts()

rankingPositionAttractionCategory
2       14
1       12
3       11
11      11
4       11
        ..
671      1
1067     1
541      1
713      1
1081     1
Name: count, Length: 658, dtype: int64

In [59]:
print(metadata['rankingPositionAttractionCategory'].max())
print(metadata['rankingPositionAttractionCategory'].min())
print(metadata['rankingPositionAttractionCategory'].mean())

1282
1
234.38796909492274


#### Attraction Category

In [60]:
metadata['attractionCategory'].value_counts()

attractionCategory
things to do                 391
tours & activities           372
transportation               218
boat tours & water sports    150
nightlife                    138
shopping                     136
food & drink                 113
spas & wellness               87
outdoor activities            80
fun & games                   46
classes & workshops           45
theater & concerts            21
museums                        5
sights & landmarks             4
nature & parks                 3
events                         1
amusement parks                1
Name: count, dtype: int64

In [61]:
# Filling missing values in Attraction Category with "no_category"
metadata['attractionCategory'] = metadata['attractionCategory'].fillna("no_category")

In [62]:
#View Results 
metadata['attractionCategory'].value_counts()

attractionCategory
things to do                 391
tours & activities           372
transportation               218
boat tours & water sports    150
nightlife                    138
shopping                     136
food & drink                 113
spas & wellness               87
outdoor activities            80
fun & games                   46
classes & workshops           45
theater & concerts            21
museums                        5
sights & landmarks             4
nature & parks                 3
events                         1
no_category                    1
amusement parks                1
Name: count, dtype: int64

#### Total Ranking Units Attraction Category


In [63]:
metadata['totalRankingUnitsAttractionCategory'].describe()

count    1811.000000
mean      985.318609
std       716.821925
min         2.000000
25%       392.000000
50%       853.000000
75%      1016.000000
max      2297.000000
Name: totalRankingUnitsAttractionCategory, dtype: float64

In [64]:
# Filling missing values in Total Raking Units Attraction Category with 2
metadata['totalRankingUnitsAttractionCategory'] = metadata['totalRankingUnitsAttractionCategory'].fillna(2).astype('int64')

In [65]:
#View Results
print(metadata['totalRankingUnitsAttractionCategory'].min())
print(metadata['totalRankingUnitsAttractionCategory'].describe())

2
count    1812.000000
mean      984.775938
std       716.996207
min         2.000000
25%       392.000000
50%       853.000000
75%      1016.000000
max      2297.000000
Name: totalRankingUnitsAttractionCategory, dtype: float64


### Final Count of missing Values
The single missing values were replaced with the minimum value showned in the data. This for not deleting all the rows that could influence results. 
For missing values bigger than 1 missing value, the replacement was considered according the needs of the information. 

In [66]:
#Number of missing values for every column
print(metadata.isnull().sum())

tripadvisorId                             0
name                                      0
location                               1243
latitude                               1243
longitude                              1243
dateFirstCollection                       0
ratingGlobal                              0
reviewCount                               0
url                                       0
attractionCategory                        0
rankingPositionAttractionCategory         0
totalRankingUnitsAttractionCategory       0
geo_id                                    0
neighborhood_extracted                 1321
pc4_original                           1243
pc4_rescued                            1812
pc4_combined                           1243
zone_final                                0
neighborhood_final                        0
dtype: int64


### Trip Advisor IDs
Metadata, Reviews and Attractions Datasets 

In [67]:
#Standarize Ids
reviews = reviews.rename(columns={'tripadvisorId': 'tripadvisor_id'})
metadata = metadata.rename(columns={'tripadvisorId': 'tripadvisor_id'})

##### Attractions List Dataset

In [68]:
import html
import unicodedata

def simplify_text(text):
    if not isinstance(text, str):
        return text
    
    #Correct HTML entities (ex: &amp; -> &)
    text = html.unescape(text)
    
    #Normalize Accents (ex: Praça -> Praca)
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    
    #Remove commas and other problematic punctuation
    text = text.replace(',', '').replace('\\', '').replace('<', '').replace('>', '')
    
    #Delete spaces and Title Case
    return text.strip().title()

#Simplify text
attractions['name'] = attractions['name'].apply(simplify_text)

#Standardize IDs to match other datasets
attractions['tripadvisor_id'] = attractions['tripadvisor_id'].astype(int)

In [69]:
#Check for null values
print(f"Total null values: {attractions['name'].isna().sum()}")

#Count empty strings or rows or spaces
empty_names = attractions[attractions['name'].str.strip() == ""]
print(f"Total empty names: {len(empty_names)}")

#View empty name values
if len(empty_names) > 0:
    print(empty_names[['url', 'tripadvisor_id']])

Total null values: 1
Total empty names: 3
                                                    url  tripadvisor_id
3567  https://www.tripadvisor.com/Attraction_Review-...        17723805
3575  https://www.tripadvisor.com/Attraction_Review-...        18276477
3593  https://www.tripadvisor.com/Attraction_Review-...        20862391


In [70]:
#check null values in metadata and reviews dataset 
#Identify IDs that have an empty string or a Null value in the name column
empty_or_null_mask = (attractions['name'].fillna('').str.strip() == "")
bad_ids = attractions[empty_or_null_mask]['tripadvisor_id'].unique()

#Check if "Bad IDs" appear in Reviews or Metadata
rescuable_in_reviews = reviews[reviews['tripadvisor_id'].isin(bad_ids)]
rescuable_in_meta = metadata[metadata['tripadvisor_id'].isin(bad_ids)]

#Show results
print(f"Total problematic IDs in Attractions: {len(bad_ids)}")
print(f"IDs with no names found in Reviews: {rescuable_in_reviews['tripadvisor_id'].nunique()}")
print(f"IDs with no names found in Metadata: {rescuable_in_meta['tripadvisor_id'].nunique()}")


Total problematic IDs in Attractions: 4
IDs with no names found in Reviews: 0
IDs with no names found in Metadata: 0


In [71]:
#Delete problematic IDs in Attractions
attractions = attractions[~(attractions['name'].fillna('').str.strip() == "")]

#Final check
print(f"Remaining attractions: {len(attractions)}")
print(f"Nulls in name column: {attractions['name'].isnull().sum()}")


Remaining attractions: 4335
Nulls in name column: 0


Problems resolved in Attraction dataset. No null values in it. 

In [72]:
#Resolve duplicate Ids

#ID and num_reviews highest to lowest
attractions = attractions.sort_values(by=['tripadvisor_id', 'num_reviews'], ascending=[True, False])

#Drop duplicates based on ID
before_count = len(attractions)
attractions = attractions.drop_duplicates(subset=['tripadvisor_id'], keep='first')
after_count = len(attractions)

print(f"IDs removed: {before_count - after_count}")

IDs removed: 0


In [73]:
#Group by name
## Sum the 'num_reviews and keep 1st 'tripadvisor_id' as reference 

attractions_grouped = attractions.groupby('name').agg({
    'tripadvisor_id': 'first',
    'url': 'first',
    'num_reviews': 'sum'
}).reset_index()

# Calculate how many duplicated merged
consolidated_count = len(attractions) - len(attractions_grouped)
print(f"Attractions consolidated: {consolidated_count}")
print(f"Unique attraction (counted): {len(attractions_grouped)}")

Attractions consolidated: 134
Unique attraction (counted): 4201


In [74]:
#Identify IDs from metadata
target_ids = set(metadata['tripadvisor_id'].astype(int))

#Filter IDS to group attractions
attractions_final = attractions_grouped[attractions_grouped['tripadvisor_id'].isin(target_ids)].copy()

#Check match
print(f"Total matched attractions: {len(attractions_final)}")

Total matched attractions: 1778


Missing 34 attractions in metadata

In [75]:
#Identify missing data

#Convert IDs to a list
target_ids = set(metadata['tripadvisor_id'].astype(int))
cleaned_ids = set(attractions_final['tripadvisor_id'])

#Find missing IDs
missing_ids = list(target_ids - cleaned_ids)

print(f"Missing IDs count: {len(missing_ids)}")

#Check missing IDS in attractions in Metadata
missing_info = metadata[metadata['tripadvisor_id'].isin(missing_ids)]
print("\nSample of Metadata of missed attractions:")
print(missing_info[['tripadvisor_id', 'name', 'ratingGlobal']].head())

Missing IDs count: 34

Sample of Metadata of missed attractions:
     tripadvisor_id                        name  ratingGlobal
1          24920617  Lisboa Espera Por Ti Tours             4
42          6105390           A Vida Portuguesa             4
62         26660496                 Indie Tours             5
122        27929036               Yourlocaljoni             5
142        33064008                      Tuk Go             5


In [76]:
#Rescue data and merge

# Make a copy of Metadata
metadata_clean = metadata.copy()

#Rename 'name' column to merge
metadata_clean = metadata_clean.rename(columns={'name': 'name_metadata'})

#Group attractions dataframe 
attractions_prep = attractions_grouped[['tripadvisor_id', 'name', 'url']].copy()
attractions_prep = attractions_prep.rename(columns={'name': 'name_attractions'})

#Join Reviews and Metadata
df_core = pd.merge(reviews, metadata_clean, on='tripadvisor_id', how='inner')

#Join df_core with Attractions
df_final = pd.merge(df_core, attractions_prep, on='tripadvisor_id', how='left')

# Use Attraction name if available or fall back to Metadata name
df_final['name'] = df_final['name_attractions'].fillna(df_final['name_metadata'])

#Remove temporary helper columns 
df_final = df_final.drop(columns=['name_metadata', 'name_attractions'])

print(f"Total Rows (Reviews): {len(df_final)}")
print(f"Total Unique Attractions: {df_final['tripadvisor_id'].nunique()}")

Total Rows (Reviews): 143656
Total Unique Attractions: 1812


In [77]:
#Top 5 attractions by number of reviews 
top_attractions = df_final['name'].value_counts().head(5)

print("Top 5 Attractions by Review:")
print(top_attractions)

#Check average rating for the whole dataset
avg_rating = df_final['ratingGlobal'].mean()
print(f"\nAverage Global Rating in reviews: {avg_rating:.2f}")

Top 5 Attractions by Review:
name
Take Lisboa                        34607
Eating Europe Food Tours Lisbon     5120
Inside Lisbon Tours                 4798
I Took A Tuk Tuk                    3948
World Experience - Lisboa           3100
Name: count, dtype: int64

Average Global Rating in reviews: 4.46


In [78]:
#Check if any reviews ended without a name after the join
orphan_reviews = df_final['name'].isnull().sum()

if orphan_reviews == 0:
    print("All 143,656 reviews have the correct name.")
else:
    print(f"{orphan_reviews} reviews have a missing name.")

All 143,656 reviews have the correct name.


TripAdvisor IDs corrected. 

### Dates and Time in Reviews

In [79]:
#Formating dates and time
reviews['reviewDate'] = pd.to_datetime(reviews['reviewDate'])
reviews['visitMonth'] = pd.to_datetime(reviews['visitMonth'])
reviews['replyDate'] = pd.to_datetime(reviews['replyDate'], errors='coerce')


In [80]:
#Format time - visitMonth
reviews['visitMonth'] = pd.to_datetime(reviews['visitMonth'])

#Calculate months from the begining
start_date = reviews['visitMonth'].min()
reviews['visitMonth_float'] = (reviews['visitMonth'].dt.year - start_date.year) * 12 + \
                              (reviews['visitMonth'].dt.month - start_date.month)

print(reviews[['visitMonth', 'visitMonth_float']].head())

  visitMonth  visitMonth_float
0 2023-09-01                75
1 2025-04-01                94
2 2025-02-01                92
3 2024-11-01                89
4 2024-12-01                90


In [81]:
#Format time - ReviewDate
date_format_iso = '%Y-%m-%d' 
reviews['reviewDate'] = pd.to_datetime(
    reviews['reviewDate'],
    format = date_format_iso, 
    errors ='coerce')

success_conv1 = reviews['reviewDate'].count()
total_rows = len(reviews)

print(f"\sucess conversion:{success_conv1} out of {total_rows}")
print(f"\success rate:{(success_conv1/total_rows) * 100:.2f}%")
print(reviews['reviewDate'].info())
print(reviews['reviewDate'].head())

\sucess conversion:143656 out of 143656
\success rate:100.00%
<class 'pandas.core.series.Series'>
RangeIndex: 143656 entries, 0 to 143655
Series name: reviewDate
Non-Null Count   Dtype         
--------------   -----         
143656 non-null  datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 1.1 MB
None
0   2023-09-27
1   2025-04-26
2   2025-02-18
3   2024-12-23
4   2024-12-13
Name: reviewDate, dtype: datetime64[ns]


In [82]:
#Format time - dateFirstCollection
date_format_iso = '%Y-%m-%d' 
reviews['dateFirstCollection'] = pd.to_datetime(
    reviews['dateFirstCollection'],
    format = date_format_iso, 
    errors ='coerce')

success_conv2 = reviews['dateFirstCollection'].count()
total_rows = len(reviews)

print(f"\sucess conversion:{success_conv2} out of {total_rows}")
print(f"\success rate:{(success_conv2/total_rows) * 100:.2f}%")
print(reviews['dateFirstCollection'].info())
print(reviews['dateFirstCollection'].head())

\sucess conversion:143656 out of 143656
\success rate:100.00%
<class 'pandas.core.series.Series'>
RangeIndex: 143656 entries, 0 to 143655
Series name: dateFirstCollection
Non-Null Count   Dtype         
--------------   -----         
143656 non-null  datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 1.1 MB
None
0   2025-08-25
1   2025-08-24
2   2025-08-24
3   2025-08-24
4   2025-08-24
Name: dateFirstCollection, dtype: datetime64[ns]


### Numerical in Reviews

##### Rating

In [83]:
# Convert ratings 
reviews['rating'] = pd.to_numeric(reviews['rating'], errors='coerce')
print(reviews['rating'].head())

0    1
1    5
2    5
3    5
4    5
Name: rating, dtype: int64


##### Contributions

In [84]:
# Convert contributions 
reviews['contributions'] = pd.to_numeric(reviews['contributions'], errors='coerce')
print(reviews['contributions'].head())

0     1
1    13
2    46
3     2
4     2
Name: contributions, dtype: int64


##### Likes Number

In [85]:
# Convert likes Number 
reviews['likesNumber'] = pd.to_numeric(reviews['likesNumber'], errors='coerce')
print(reviews['likesNumber'].head(10))

0    0
1    0
2    0
3    0
4    0
5    0
6    0
7    0
8    0
9    0
Name: likesNumber, dtype: int64


### Dates and Time in Metadata Dataset

##### Review Date

In [86]:
# Convert review date
reviews['reviewDate'] = pd.to_datetime(reviews['reviewDate'])
print(reviews['reviewDate'].head())

0   2023-09-27
1   2025-04-26
2   2025-02-18
3   2024-12-23
4   2024-12-13
Name: reviewDate, dtype: datetime64[ns]


##### Reply Date 

In [87]:
# Convert reply date
reviews['replyDate'] = pd.to_datetime(reviews['replyDate'], errors='coerce')
print(reviews['replyDate'].head())

0   2023-01-01
1   2025-04-29
2   2025-02-26
3   2024-12-26
4   2024-12-26
Name: replyDate, dtype: datetime64[ns]


##### Date First Collection

In [88]:
#Format time - Date First Collection 
metadata['dateFirstCollection'] = pd.to_datetime(metadata['dateFirstCollection'])

metadata['dateFirstCollection'].head()

0   2025-08-27 12:53:09
1   2025-08-27 12:53:53
2   2025-08-27 12:54:40
3   2025-08-27 12:55:27
4   2025-08-27 12:56:16
Name: dateFirstCollection, dtype: datetime64[ns]

In [89]:
# Completing Data for latter analysis 
metadata['collection_year'] = metadata['dateFirstCollection'].dt.year
metadata['collection_month'] = metadata['dateFirstCollection'].dt.month
metadata['collection_hour'] = metadata['dateFirstCollection'].dt.hour
metadata['day_of_week'] = metadata['dateFirstCollection'].dt.day_name()

#Showing results
print(metadata['collection_year'].head())
print(metadata['collection_month'].head())
print(metadata['collection_hour'].head())
print(metadata['day_of_week'].head())


0    2025
1    2025
2    2025
3    2025
4    2025
Name: collection_year, dtype: int32
0    8
1    8
2    8
3    8
4    8
Name: collection_month, dtype: int32
0    12
1    12
2    12
3    12
4    12
Name: collection_hour, dtype: int32
0    Wednesday
1    Wednesday
2    Wednesday
3    Wednesday
4    Wednesday
Name: day_of_week, dtype: object


In [90]:
#Timezone identification, if any
metadata['dateFirstCollection'] = pd.to_datetime(metadata['dateFirstCollection'], utc=True).dt.tz_localize(None)
print(metadata['dateFirstCollection'])


0      2025-08-27 12:53:09
1      2025-08-27 12:53:53
2      2025-08-27 12:54:40
3      2025-08-27 12:55:27
4      2025-08-27 12:56:16
               ...        
1807   2025-09-01 15:11:21
1808   2025-09-01 15:12:08
1809   2025-09-01 15:12:47
1810   2025-09-01 15:13:33
1811   2025-09-01 15:14:24
Name: dateFirstCollection, Length: 1812, dtype: datetime64[ns]


### Numerical in Metadata 
No Data to format

### Format Types at the end

In [91]:
#Type of columns in metadata
print(reviews.dtypes)

title                          object
text                           object
rating                          int64
reviewer                       object
visitMonth             datetime64[ns]
country                        object
visitType                      object
contributions                   int64
likesNumber                     int64
replyText                      object
replyDate              datetime64[ns]
reviewDate             datetime64[ns]
name                           object
tripadvisor_id                  int64
dateFirstCollection    datetime64[ns]
country_inter                  object
country_final                  object
country_clean                  object
visitMonth_float                int32
dtype: object


In [92]:
#Type of columns in metadata after transformatio
print(metadata.dtypes)

tripadvisor_id                                  int64
name                                           object
location                                       object
latitude                                      float64
longitude                                     float64
dateFirstCollection                    datetime64[ns]
ratingGlobal                                    int64
reviewCount                                     int64
url                                            object
attractionCategory                             object
rankingPositionAttractionCategory               int64
totalRankingUnitsAttractionCategory             int64
geo_id                                         object
neighborhood_extracted                         object
pc4_original                                   object
pc4_rescued                                    object
pc4_combined                                   object
zone_final                                     object
neighborhood_final          

In [93]:
#Type of columns in attractions
print(attractions.dtypes)

name              object
url               object
tripadvisor_id     int64
num_reviews        int64
dtype: object


### Delete Duplicates

In [94]:
#Check for exact duplicate rows in the reviews dataset
exact_duplicates = reviews.duplicated().sum()
print(f"Exact duplicate rows in reviews: {exact_duplicates}")

#Check for reviews that might be duplicates based on key columns
key_cols = ['reviewer', 'text', 'reviewDate', 'tripadvisor_id']
logical_duplicates = reviews.duplicated(subset=key_cols).sum()
print(f"Logical duplicates (same reviewer/text/date): {logical_duplicates}")

#If duplicates are found, remove them
if logical_duplicates > 0:
    reviews = reviews.drop_duplicates(subset=key_cols).reset_index(drop=True)
    print("Duplicates removed.")

Exact duplicate rows in reviews: 0
Logical duplicates (same reviewer/text/date): 3
Duplicates removed.


In [95]:
#Check if tripadvisor Id is repeated in metadata
metadata_duplicates = metadata['tripadvisor_id'].duplicated().sum()
print(f"Duplicate tripadvisorIds in metadata: {metadata_duplicates}")

# If duplicates exist, investigate why or keep only the first occurrence
if metadata_duplicates > 0:
    metadata = metadata.drop_duplicates(subset=['tripadvisor_id']).reset_index(drop=True)
    print("Metadata duplicates removed.")

Duplicate tripadvisorIds in metadata: 0


### Text

In [96]:
#Check for duplicated content 
print(reviews.duplicated().sum())
print(metadata.duplicated().sum())

0
0


In [97]:
#Basic text cleaning 
def clean_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)  #Remove html 
    text = re.sub(r'[^\w\s]', '', text) #Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip() #Remove whitespace
    return text

reviews['text_clean'] = reviews['text'].apply(clean_text)
reviews['title_clean'] = reviews['title'].apply(clean_text)
reviews['replyText_clean'] = reviews['replyText'].apply(clean_text)

print(reviews['text_clean'].head())
print(reviews['title_clean'].head())
print(reviews['replyText_clean'].head())

0    rented 4 cars from car hire in leclerc superma...
1    the service was brilliant and francisco couldn...
2    found this spot on tripadvisor we were in the ...
3    it is truly remarkable that your culinary expe...
4    the perfect place if you want to have a drink ...
Name: text_clean, dtype: object
0    location leclerc car hire carcassonne aboid
1                                would recommend
2                            delicious cocktails
3       a place where you can enjoy every flavor
4                              a drink in lisboa
Name: title_clean, dtype: object
0                                        no_reply_text
1    dear 796loist thank you so much for your kind ...
2    dear perryla thank you for stopping by and for...
3    dear attilaerikf2024 thank you so much for you...
4    dear attillau thank you so much for your lovel...
Name: replyText_clean, dtype: object


### Merging Data


In [98]:
#Create a version with reviews columns 
df_reviews_master = reviews[[
    'tripadvisor_id', 'name','title', 'text', 'rating', 
    'country_final', 'visitMonth', 'visitType', 'contributions',
    'likesNumber','replyText', 'replyDate', 'reviewDate', 'dateFirstCollection'
    ]].copy()


In [99]:
#Change name in columns

new_names_reviews = {
    'country_final': 'visitorCountry',
    'rating': 'reviewStars', 
}

# Apply the change
df_reviews_master = df_reviews_master.rename(columns=new_names_reviews)

# Check the results
print(df_reviews_master.columns)

Index(['tripadvisor_id', 'name', 'title', 'text', 'reviewStars',
       'visitorCountry', 'visitMonth', 'visitType', 'contributions',
       'likesNumber', 'replyText', 'replyDate', 'reviewDate',
       'dateFirstCollection'],
      dtype='object')


In [100]:
#Create a version with metadata columns 
df_metadata_master = metadata[[
    'tripadvisor_id', 'name', 'dateFirstCollection', 'ratingGlobal',
    'reviewCount', 'attractionCategory', 'rankingPositionAttractionCategory',
    'totalRankingUnitsAttractionCategory', 'neighborhood_final', 'collection_year',
    'collection_month', 'collection_hour', 'day_of_week' 
    ]].copy()

In [101]:
#Change name in columns

new_names_metadata = {
    'neighborhood_final': 'lisbonLocation',
    'day_of_week': 'collection_day_of_week'
}

# Apply the change
df_metadata_master = df_metadata_master.rename(columns=new_names_metadata)

# Check the results
print(df_metadata_master.columns)

Index(['tripadvisor_id', 'name', 'dateFirstCollection', 'ratingGlobal',
       'reviewCount', 'attractionCategory',
       'rankingPositionAttractionCategory',
       'totalRankingUnitsAttractionCategory', 'lisbonLocation',
       'collection_year', 'collection_month', 'collection_hour',
       'collection_day_of_week'],
      dtype='object')


### Exporting Data

In [102]:
print(df_reviews_master.isnull().sum())
print(df_metadata_master.isnull().sum())

tripadvisor_id         0
name                   0
title                  0
text                   0
reviewStars            0
visitorCountry         0
visitMonth             0
visitType              0
contributions          0
likesNumber            0
replyText              0
replyDate              0
reviewDate             0
dateFirstCollection    0
dtype: int64
tripadvisor_id                         0
name                                   0
dateFirstCollection                    0
ratingGlobal                           0
reviewCount                            0
attractionCategory                     0
rankingPositionAttractionCategory      0
totalRankingUnitsAttractionCategory    0
lisbonLocation                         0
collection_year                        0
collection_month                       0
collection_hour                        0
collection_day_of_week                 0
dtype: int64


In [103]:
#Export to CSV
# index=False is important so you don't get an extra "Unnamed: 0" column later
df_metadata_master.to_csv('Metadata_DP_Final.csv', index=False)
df_reviews_master.to_csv('Reviews_DP_Final.csv', index=False)

print("File saved!")

File saved!
